# Decomposizione ai valori singolari (SVD)

La **Singular Value Decomposition** è una delle fattorizzazioni più importanti dell'algebra lineare numerica. È definita per ogni matrice, anche rettangolare o di rango non massimo, e permette di descrivere in modo trasparente:

- l'azione geometrica di una matrice;
- rango, nucleo e immagine;
- norme e condizionamento;
- approssimazioni a rango basso e compressione dei dati.

## Richiamo: ortogonalità

Due vettori $x,y\in\mathbb{R}^n$ sono ortogonali se $x^Ty=0$. Un insieme di vettori è **ortonormale** se i vettori sono ortogonali tra loro e hanno norma euclidea unitaria.

Una matrice quadrata $Q$ è ortogonale quando le sue colonne formano una base ortonormale, cioè

$$
Q^TQ=QQ^T=I, \qquad Q^{-1}=Q^T.
$$

Le trasformazioni ortogonali conservano prodotto scalare, norma 2, distanze e angoli:

$$
\|Qx\|_2=\|x\|_2.
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

theta = np.pi / 4
Q = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
x = np.array([2.0, -1.0])

print('Q^T Q =')
print(np.round(Q.T @ Q, 12))
print(f'||x||_2  = {np.linalg.norm(x):.6f}')
print(f'||Qx||_2 = {np.linalg.norm(Q @ x):.6f}')

## Teorema di esistenza della SVD

Per ogni matrice reale $A\in\mathbb{R}^{m\times n}$ esistono due matrici ortogonali

$$
U\in\mathbb{R}^{m\times m}, \qquad
V\in\mathbb{R}^{n\times n},
$$

e una matrice rettangolare diagonale $\Sigma\in\mathbb{R}^{m\times n}$ tali che

$$
\boxed{A=U\Sigma V^T}.
$$

Ponendo $p=\min(m,n)$, sulla diagonale di $\Sigma$ compaiono i **valori singolari** ordinati:

$$
\sigma_1\geq\sigma_2\geq\cdots\geq\sigma_p\geq0.
$$

## Dimensioni delle matrici

Nella SVD completa le dimensioni sono:

| matrice | dimensione | significato |
|:--:|:--:|:--|
| $A$ | $m\times n$ | matrice di partenza |
| $U$ | $m\times m$ | vettori singolari sinistri |
| $\Sigma$ | $m\times n$ | valori singolari sulla diagonale |
| $V$ | $n\times n$ | vettori singolari destri |

Le colonne di $U=(u_1,\ldots,u_m)$ e di $V=(v_1,\ldots,v_n)$ sono basi ortonormali rispettivamente di $\mathbb{R}^m$ e $\mathbb{R}^n$.

La figura seguente, tratta dalle slide, evidenzia le dimensioni dei fattori nella SVD completa.

```{figure} immagini_sorgente/svd_dimensioni.png
---
width: 100%
align: center
---
```
<p align="center">
  <img src="immagini_sorgente/svd_dimensioni.png" width="700">
</p>

## SVD compatta

Se $r=\operatorname{rank}(A)$, allora esattamente $r$ valori singolari sono positivi:

$$
\sigma_1\geq\cdots\geq\sigma_r>0, \qquad
\sigma_{r+1}=\cdots=\sigma_p=0.
$$

Eliminando le colonne associate ai valori singolari nulli si ottiene la **SVD compatta**:

$$
A=U_r\Sigma_rV_r^T,
$$

dove $U_r\in\mathbb{R}^{m\times r}$, $\Sigma_r\in\mathbb{R}^{r\times r}$ e $V_r\in\mathbb{R}^{n\times r}$. Questa forma contiene tutta l'informazione necessaria a ricostruire $A$.

## Vettori singolari

Per ogni valore singolare positivo valgono le relazioni fondamentali

$$
Av_i=\sigma_i u_i, \qquad
A^Tu_i=\sigma_i v_i.
$$

Quindi $A$ trasforma il vettore unitario $v_i$ nel vettore $u_i$, moltiplicandone la lunghezza per $\sigma_i$. Le direzioni $v_i$ sono le direzioni principali nel dominio; le direzioni $u_i$ sono le corrispondenti direzioni nel codominio.

## Interpretazione geometrica

La trasformazione $x\mapsto Ax$ può essere letta in tre passaggi:

$$
x \xrightarrow{\,V^T\,} V^Tx
\xrightarrow{\,\Sigma\,} \Sigma V^Tx
\xrightarrow{\,U\,} U\Sigma V^Tx.
$$

1. $V^T$ ruota o riflette il sistema di riferimento;
2. $\Sigma$ dilata o contrae lungo direzioni ortogonali;
3. $U$ applica una seconda rotazione o riflessione.

In due dimensioni, la circonferenza unitaria viene trasformata in un'ellisse i cui semiassi hanno lunghezza $\sigma_1$ e $\sigma_2$.

In [ ]:
# Trasformazione della circonferenza unitaria in un'ellisse
A_geo = np.array([[2.0, 1.0],
                  [0.5, 1.5]])
t = np.linspace(0, 2*np.pi, 400)
cerchio = np.vstack((np.cos(t), np.sin(t)))
ellisse = A_geo @ cerchio
U_geo, s_geo, VT_geo = np.linalg.svd(A_geo)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(cerchio[0], cerchio[1], color='tab:blue')
ax[0].set_title('Dominio: circonferenza unitaria')
ax[1].plot(ellisse[0], ellisse[1], color='tab:red')
ax[1].set_title('Codominio: ellisse')

for a in ax:
    a.axhline(0, color='0.7', lw=0.8)
    a.axvline(0, color='0.7', lw=0.8)
    a.set_aspect('equal')
    a.grid(alpha=0.25)

for i, colore in enumerate(['tab:green', 'tab:purple']):
    asse = s_geo[i] * U_geo[:, i]
    ax[1].quiver(0, 0, asse[0], asse[1], angles='xy',
                 scale_units='xy', scale=1, color=colore,
                 label=fr'$\sigma_{i+1}u_{i+1}$')
ax[1].legend()
plt.tight_layout()
plt.show()
print('Valori singolari:', np.round(s_geo, 4))

## Relazione con autovalori e autovettori

Dalla SVD segue

$$
A^TA=V\Sigma^T\Sigma V^T, \qquad
AA^T=U\Sigma\Sigma^T U^T.
$$

Pertanto:

$$
A^TAv_i=\sigma_i^2v_i, \qquad
AA^Tu_i=\sigma_i^2u_i.
$$

I valori singolari sono quindi le radici quadrate degli autovalori non negativi di $A^TA$:

$$
\sigma_i=\sqrt{\lambda_i(A^TA)}.
$$

> Questa relazione è utile teoricamente, ma in generale non conviene calcolare numericamente la SVD formando esplicitamente $A^TA$: il condizionamento viene elevato al quadrato e si può perdere accuratezza.

La diagonalizzazione di $A^TA$ mediante i vettori singolari destri è rappresentata nello schema delle slide.

```{figure} immagini_sorgente/svd_ata.png
---
width: 100%
align: center
---
```
<p align="center">
  <img src="immagini_sorgente/svd_ata.png" width="700">
</p>

In [ ]:
# Esempio di SVD di una matrice rettangolare 3 x 2
A = np.array([[3.0, 1.0],
              [1.0, 3.0],
              [1.0, 1.0]])

# full_matrices=False restituisce la SVD ridotta
U, s, VT = np.linalg.svd(A, full_matrices=False)
Sigma = np.diag(s)
A_ricostruita = U @ Sigma @ VT

print('U =\n', np.round(U, 4))
print('Valori singolari =', np.round(s, 4))
print('V^T =\n', np.round(VT, 4))
print('Errore di ricostruzione =', np.linalg.norm(A-A_ricostruita))
print('U^T U =\n', np.round(U.T @ U, 12))
print('V^T V =\n', np.round(VT @ VT.T, 12))

## Rango, immagine e nucleo

Se $A$ ha rango $r$, la SVD rende immediatamente visibili i suoi sottospazi fondamentali:

$$
\operatorname{Im}(A)=\operatorname{span}\{u_1,\ldots,u_r\},
$$

$$
\operatorname{Im}(A^T)=\operatorname{span}\{v_1,\ldots,v_r\},
$$

$$
\ker(A)=\operatorname{span}\{v_{r+1},\ldots,v_n\}.
$$

In aritmetica floating point non si verifica quasi mai se $\sigma_i$ è esattamente zero: si confronta con una tolleranza dipendente da $\sigma_1$, dalle dimensioni della matrice e dalla precisione di macchina.

## Decomposizione diadica

La SVD compatta può essere scritta come somma di matrici di rango uno:

$$
\boxed{A=\sum_{i=1}^{r}\sigma_i u_i v_i^T}.
$$

Ogni termine $\sigma_i u_i v_i^T$ descrive una componente indipendente della matrice. Le componenti sono ordinate dalla più importante alla meno importante in base alla grandezza dei valori singolari.

In [ ]:
# Ricostruzione come somma di diadi
A_somma = np.zeros_like(A)
for i in range(len(s)):
    diade = s[i] * np.outer(U[:, i], VT[i, :])
    A_somma += diade
    print(f'Dopo {i+1} termine/i: errore di Frobenius = '
          f'{np.linalg.norm(A-A_somma, ord="fro"):.6f}')

## SVD troncata e approssimazione a rango basso

Se conserviamo soltanto i primi $k<r$ termini otteniamo

$$
A_k=\sum_{i=1}^{k}\sigma_i u_i v_i^T.
$$

Il teorema di Eckart-Young-Mirsky afferma che $A_k$ è la migliore approssimazione di rango al più $k$ sia in norma 2 sia in norma di Frobenius:

$$
\|A-A_k\|_2=\sigma_{k+1},
$$

$$
\|A-A_k\|_F=\sqrt{\sum_{i=k+1}^{r}\sigma_i^2}.
$$

Questa proprietà è alla base di compressione, riduzione dimensionale e filtraggio del rumore.

In [ ]:
# Verifica numerica del teorema di Eckart-Young-Mirsky
B = np.array([[4., 1., 1., 0.],
              [1., 3., 0., 1.],
              [1., 0., 2., 1.],
              [0., 1., 1., 1.]])
Ub, sb, VTb = np.linalg.svd(B, full_matrices=False)

print('Valori singolari:', np.round(sb, 4))
for k in range(1, len(sb)):
    Bk = (Ub[:, :k] * sb[:k]) @ VTb[:k, :]
    err2 = np.linalg.norm(B-Bk, 2)
    errF = np.linalg.norm(B-Bk, 'fro')
    teoricoF = np.sqrt(np.sum(sb[k:]**2))
    print(f'k={k}: errore 2 = {err2:.4f} (sigma_{k+1}={sb[k]:.4f}), '
          f'errore F = {errF:.4f} (teorico={teoricoF:.4f})')

## Norme e numero di condizionamento

La SVD consente di calcolare direttamente alcune quantità fondamentali:

$$
\|A\|_2=\sigma_1,
$$

$$
\|A\|_F=\sqrt{\sigma_1^2+\cdots+\sigma_r^2}.
$$

Se $A$ è quadrata e invertibile,

$$
\|A^{-1}\|_2=\frac{1}{\sigma_n}, \qquad
\kappa_2(A)=\frac{\sigma_1}{\sigma_n}.
$$

Un valore singolare minimo molto piccolo indica che esiste una direzione fortemente contratta da $A$ e che la matrice è vicina a essere singolare.

In [ ]:
C = np.array([[1.000, 2.000],
              [0.499, 1.001]])
sc = np.linalg.svd(C, compute_uv=False)

print('Valori singolari:', sc)
print('Norma 2 dalla SVD:', sc[0])
print('Norma 2 con NumPy:', np.linalg.norm(C, 2))
print('Condizionamento dalla SVD:', sc[0] / sc[-1])
print('Condizionamento con NumPy:', np.linalg.cond(C, 2))

## Pseudoinversa mediante SVD

Dalla SVD compatta $A=U_r\Sigma_rV_r^T$ si definisce la pseudoinversa di Moore-Penrose:

$$
A^+=V_r\Sigma_r^{-1}U_r^T,
$$

dove

$$
\Sigma_r^{-1}=\operatorname{diag}
\left(\frac1{\sigma_1},\ldots,\frac1{\sigma_r}\right).
$$

Se $A$ è quadrata e invertibile, allora $A^+=A^{-1}$. Per matrici di rango non massimo, i valori singolari nulli non vengono invertiti. In pratica si usa una tolleranza per decidere quali valori singolari trattare come nulli.

In [ ]:
# Pseudoinversa di una matrice di rango non massimo
D = np.array([[1., 2., 3.],
              [2., 4., 6.]])
Ud, sd, VTd = np.linalg.svd(D, full_matrices=False)
tol = max(D.shape) * np.finfo(float).eps * sd[0]
sd_inv = np.array([1/sigma if sigma > tol else 0.0 for sigma in sd])
D_piu = VTd.T @ np.diag(sd_inv) @ Ud.T

print('Valori singolari:', sd)
print('Rango numerico:', np.sum(sd > tol))
print('||D D+ D - D||_F =', np.linalg.norm(D @ D_piu @ D-D))
print('Confronto con np.linalg.pinv:', np.linalg.norm(D_piu-np.linalg.pinv(D)))

## Aspetti numerici

- Per una matrice densa $m\times n$, il costo della SVD completa è dell'ordine di $mn\min(m,n)$.
- `np.linalg.svd(A, full_matrices=False)` calcola la forma ridotta ed evita di memorizzare colonne non necessarie.
- Per matrici molto grandi e sparse si calcolano spesso soltanto i primi valori e vettori singolari con metodi iterativi.
- Il rango numerico dipende dalla tolleranza: valori singolari molto piccoli possono essere indistinguibili da zero in precisione finita.
- Formare esplicitamente $A^TA$ può peggiorare l'accuratezza numerica; gli algoritmi standard lavorano direttamente su $A$, tipicamente tramite bidiagonalizzazione.

## Approximazione mediante diadi

La decomposizione diadica ordina le componenti di $A$ in base alla loro importanza:

$$
A=\underbrace{\sigma_1u_1v_1^T}_{D_1}+
  \underbrace{\sigma_2u_2v_2^T}_{D_2}+\cdots+
  \underbrace{\sigma_ru_rv_r^T}_{D_r}.
$$

Ogni diade $D_i=\sigma_i u_iv_i^T$ ha rango uno. Le somme parziali

$$
A_k=D_1+\cdots+D_k=\sum_{i=1}^{k}\sigma_i u_i v_i^T
$$

hanno rango al più $k$ e forniscono approssimazioni via via più accurate di $A$. Il residuo dopo $k$ termini è

$$
R_k=A-A_k=\sum_{i=k+1}^{r}\sigma_i u_i v_i^T.
$$

Poiché $\sigma_1\geq\sigma_2\geq\cdots$, si conservano per prime le direzioni che producono la maggiore variazione.

Per rendere visibile l'effetto della SVD usiamo una vera immagine in **scala di grigi**, ottenuta dalla fotografia di un fiore inclusa tra i dati di esempio di scikit-learn. In questo modo l'immagine è rappresentata direttamente da una sola matrice.

```{figure} immagini_sorgente/flower_gray.png
---
width: 100%
align: center
---
```
<p align="center">
  <img src="immagini_sorgente/flower_gray.png" width="700">
</p>

In [ ]:
# Immagine in scala di grigi: ogni pixel è un elemento della matrice
immagine = plt.imread('immagini_sorgente/flower_gray.png')
if immagine.max() > 1:
    immagine = immagine / 255.0

# Sottocampionamento per rendere l'esempio rapido anche su un portatile
immagine = immagine[::2, ::2]
U_img, s_img, VT_img = np.linalg.svd(immagine, full_matrices=False)

def approssimazione_svd(k):
    return (U_img[:, :k] * s_img[:k]) @ VT_img[:k, :]

diadi_img = [s_img[i] * np.outer(U_img[:, i], VT_img[i, :])
              for i in range(3)]
ranghi = [1, 5, 20, 50]
approssimazioni_img = [approssimazione_svd(k) for k in ranghi]

fig, ax = plt.subplots(2, 4, figsize=(13, 8))
ax[0, 0].imshow(immagine, cmap='gray', vmin=0, vmax=1)
ax[0, 0].set_title(f'Originale: {immagine.shape[0]} x {immagine.shape[1]}')
for i in range(3):
    ax[0, i+1].imshow(diadi_img[i], cmap='gray')
    ax[0, i+1].set_title(fr'Diade $D_{i+1}$')

for a, Ak, k in zip(ax[1], approssimazioni_img, ranghi):
    a.imshow(np.clip(Ak, 0, 1), cmap='gray', vmin=0, vmax=1)
    a.set_title(fr'Approssimazione $A_{{{k}}}$')

for a in ax.ravel():
    a.axis('off')
plt.suptitle('Diadi e approssimazioni SVD a rango crescente', fontsize=15)
plt.tight_layout()
plt.show()

### Qualità dell'approssimazione

L'approssimazione $A_k$ non è soltanto una possibile matrice di rango $k$: è la **migliore** tra tutte le matrici di rango al più $k$. Gli errori sono

$$
\|A-A_k\|_2=\sigma_{k+1}, \qquad
\|A-A_k\|_F=\sqrt{\sigma_{k+1}^2+\cdots+\sigma_r^2}.
$$

La frazione di energia conservata dai primi $k$ termini è

$$
E_k=\frac{\sigma_1^2+\cdots+\sigma_k^2}
{\sigma_1^2+\cdots+\sigma_r^2}.
$$

Se i valori singolari decrescono rapidamente, pochi termini descrivono bene la matrice. Se decadono lentamente, è necessario conservare molte diadi.

In [ ]:
energia = np.cumsum(s_img**2) / np.sum(s_img**2)
m, n = immagine.shape
ranghi_tabella = [1, 5, 10, 20, 50, 100]

print(' k   errore relativo F   energia conservata   dati rispetto all originale')
for k in ranghi_tabella:
    Ak = approssimazione_svd(k)
    errore_rel = np.linalg.norm(immagine-Ak, 'fro') / np.linalg.norm(immagine, 'fro')
    frazione_dati = k * (m+n+1) / (m*n)
    print(f'{k:3d}       {errore_rel:8.4f}             '
          f'{energia[k-1]:8.4f}                 {frazione_dati:8.4f}')

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
indici = np.arange(1, min(120, len(s_img))+1)
ax[0].semilogy(indici, s_img[:len(indici)], color='tab:blue')
ax[0].set(title='Decadimento dei valori singolari', xlabel='$i$', ylabel=r'$\sigma_i$')
ax[1].plot(indici, energia[:len(indici)], color='tab:green')
ax[1].set(title='Energia cumulativa', xlabel='$k$', ylabel='$E_k$', ylim=(0, 1.02))
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretazione come compressione

Una matrice $A\in\mathbb{R}^{m\times n}$ richiede $mn$ numeri. Per memorizzare la sua approssimazione $A_k=U_k\Sigma_kV_k^T$ sono sufficienti

$$
mk+k+nk=k(m+n+1)
$$

numeri. Si ottiene una vera compressione quando

$$
k(m+n+1)<mn.
$$

La scelta di $k$ realizza quindi un compromesso: valori piccoli producono maggiore compressione, mentre valori grandi riducono l'errore di approssimazione.

# PCA mediante SVD

La **Principal Component Analysis** (PCA) cerca nuove direzioni ortogonali lungo le quali i dati presentano la massima variabilità. Può essere calcolata direttamente mediante la SVD della matrice dei dati centrati.

Sia $X\in\mathbb{R}^{N\times p}$ una matrice in cui le righe rappresentano gli oggetti osservati e le colonne le variabili. Prima di applicare la PCA si sottrae da ogni colonna la propria media:

$$
X_c=X-\mathbf{1}\mu^T.
$$

La centratura è essenziale: la PCA deve descrivere le variazioni rispetto al centro dei dati, non la loro posizione rispetto all'origine.

## Legame tra PCA e SVD

Calcoliamo la SVD della matrice centrata:

$$
X_c=U\Sigma V^T.
$$

La matrice di covarianza è

$$
C=\frac{1}{N-1}X_c^TX_c
 =V\frac{\Sigma^2}{N-1}V^T.
$$

Di conseguenza:

- le colonne di $V$ sono le **direzioni principali**;
- le varianze lungo tali direzioni sono $\lambda_i=\sigma_i^2/(N-1)$;
- le coordinate dei dati nel nuovo sistema, dette **score**, sono

$$
Z=X_cV=U\Sigma.
$$

## Esempio: il dataset Iris

Il dataset contiene 150 fiori appartenenti a tre specie. Ogni fiore è descritto da quattro variabili: lunghezza e larghezza del sepalo, lunghezza e larghezza del petalo.

Calcoliamo la PCA usando soltanto `numpy.linalg.svd`. Le specie non vengono usate nel calcolo: serviranno unicamente per colorare il grafico finale.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data
y = iris.target
nomi_variabili = iris.feature_names

media = X.mean(axis=0)
Xc = X-media
U_pca, s_pca, VT_pca = np.linalg.svd(Xc, full_matrices=False)

varianza = s_pca**2 / (X.shape[0]-1)
varianza_spiegata = varianza / varianza.sum()
score = Xc @ VT_pca.T       # equivalente a U_pca * s_pca

print('Dimensione di X:', X.shape)
print('Media delle variabili:', np.round(media, 3))
print('Valori singolari:', np.round(s_pca, 3))
print('Varianza spiegata:', np.round(varianza_spiegata, 4))
print('Varianza spiegata dalle prime due componenti:',
      f'{100*varianza_spiegata[:2].sum():.2f}%')

## Proiezione sulle prime due componenti

Per ridurre i dati da quattro a due dimensioni conserviamo soltanto le prime due colonne di $V$:

$$
Z_2=X_cV_2\in\mathbb{R}^{N\times2}.
$$

Ogni riga di $Z_2$ contiene le coordinate di un fiore rispetto alle prime due componenti principali.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

for classe, nome, colore in zip(range(3), iris.target_names,
                                 ['tab:blue', 'tab:orange', 'tab:green']):
    selezione = y == classe
    ax[0].scatter(score[selezione, 0], score[selezione, 1],
                  label=nome, color=colore, alpha=0.75)
ax[0].set_xlabel(f'PC1 ({100*varianza_spiegata[0]:.1f}% della varianza)')
ax[0].set_ylabel(f'PC2 ({100*varianza_spiegata[1]:.1f}% della varianza)')
ax[0].set_title('Iris proiettato sulle prime due componenti')
ax[0].legend()
ax[0].grid(alpha=0.25)

componenti = np.arange(1, len(varianza_spiegata)+1)
ax[1].bar(componenti, varianza_spiegata, color='tab:blue', alpha=0.75,
          label='singola componente')
ax[1].plot(componenti, np.cumsum(varianza_spiegata), 'o-',
           color='tab:red', label='cumulativa')
ax[1].set_xticks(componenti)
ax[1].set_ylim(0, 1.05)
ax[1].set_xlabel('Componente principale')
ax[1].set_ylabel('Frazione di varianza')
ax[1].set_title('Varianza spiegata')
ax[1].legend()
ax[1].grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.show()

## Interpretazione delle componenti

Gli elementi di $v_i$ sono detti **loadings** e misurano il contributo delle variabili originali alla componente principale $i$:

$$
z_i=X_cv_i.
$$

Il segno di una componente principale è arbitrario: sostituire contemporaneamente $v_i$ con $-v_i$ e $u_i$ con $-u_i$ non cambia la fattorizzazione né l'informazione rappresentata.

In [ ]:
larghezza = 0.35
posizioni = np.arange(len(nomi_variabili))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(posizioni-larghezza/2, VT_pca[0], larghezza, label='PC1')
ax.bar(posizioni+larghezza/2, VT_pca[1], larghezza, label='PC2')
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(posizioni, nomi_variabili, rotation=20, ha='right')
ax.set_ylabel('Loading')
ax.set_title('Contributo delle variabili alle prime due componenti')
ax.legend()
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## Ricostruzione con un numero ridotto di componenti

Dopo aver conservato $k$ componenti, i dati centrati si ricostruiscono mediante

$$
X_{c,k}=Z_kV_k^T=U_k\Sigma_kV_k^T.
$$

Per tornare alle variabili originali bisogna aggiungere nuovamente la media:

$$
\widehat X_k=X_{c,k}+\mathbf{1}\mu^T.
$$

La PCA è quindi un'applicazione diretta della SVD troncata alla matrice dei dati centrati.

In [ ]:
print(' k   varianza conservata   errore relativo di ricostruzione')
for k in range(1, X.shape[1]+1):
    Zk = score[:, :k]
    X_ricostruita = Zk @ VT_pca[:k, :] + media
    errore = np.linalg.norm(X-X_ricostruita, 'fro') / np.linalg.norm(Xc, 'fro')
    print(f'{k:2d}          {np.sum(varianza_spiegata[:k]):8.4f}'
          f'                    {errore:8.4f}')

### Una scelta importante: centrare o standardizzare?

La centratura è sempre necessaria. Se le variabili hanno unità di misura o scale molto diverse, è spesso opportuno anche dividerle per la deviazione standard. In tal caso la PCA viene applicata alla matrice standardizzata e non alla sola matrice centrata. La scelta dipende dal significato dei dati.

## In sintesi

La fattorizzazione

$$
A=U\Sigma V^T
$$

separa l'azione di una matrice in trasformazioni ortogonali e dilatazioni. I valori singolari misurano l'importanza delle diverse direzioni e permettono di leggere direttamente rango, norme, condizionamento e qualità delle approssimazioni a rango basso.